In [ ]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.3 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
import pandas as pd

In [ ]:
embeddings = np.load("anime_embeddings.npy")

In [ ]:
embeddings_np = np.array(embeddings).astype('float32')

In [ ]:
faiss.normalize_L2(embeddings_np)

In [ ]:
dimensions = embeddings_np.shape[1]

In [ ]:
dimensions

384

In [ ]:
index = faiss.IndexFlatIP(dimensions)

In [ ]:
index.add(embeddings_np)

In [ ]:
index.ntotal

13612

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
faiss.write_index(index, '/content/drive/MyDrive/anime_project/anime.index')

Test

In [ ]:
index = faiss.read_index('/content/drive/MyDrive/anime_project/anime.index')

In [ ]:
query_embedding = embeddings_np[1:2]

In [ ]:
distances, indices = index.search(query_embedding, 5)

In [ ]:
anime_df=pd.read_csv('/content/drive/MyDrive/anime_recommendation_project/data/anime_clean.csv')

In [ ]:
print("Query anime:", anime_df.iloc[1]['title'])

Query anime: Trigun


In [ ]:
print("\nTop 5 similar:")
for i, idx in enumerate(indices[0]):
    print(anime_df.iloc[idx]['title'], '-', round(float(distances[0][i]), 3))


Top 5 similar:
Trigun - 1.0
Trigun Stampede - 0.806
Trigun: Badlands Rumble - 0.769
Trigun Stargaze - 0.668
Gatchaman Crowds Insight - 0.491


In [42]:
from difflib import SequenceMatcher

def get_recommendations(title, n=5):
    query_row = anime_df[anime_df['title'].str.contains(title, case=False)].iloc[0]
    query_idx = query_row.name
    query_title = query_row['title']

    query_embedding = embeddings_np[query_idx:query_idx+1].copy()
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, n + 20)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx == query_idx:  # skip itself
            continue

        candidate_title = anime_df.iloc[idx]['title']
        similarity_score = round(float(distances[0][i]), 3)

        query_root = query_title.lower().split(':')[0].strip()
        candidate_root = candidate_title.lower().split(':')[0].strip()
        if query_root in candidate_title.lower() or candidate_root in query_title.lower():
            continue

        ratio = SequenceMatcher(None, query_title.lower(), candidate_title.lower()).ratio()
        if ratio > 0.6:
            continue

        results.append({
            'title': candidate_title,
            'similarity': similarity_score
        })

        if len(results) == n:
            break

    return results

In [ ]:
get_recommendations('Trigun')

[{'title': 'Gatchaman Crowds Insight', 'similarity': 0.491},
 {'title': 'Hokuto no Ken Movie', 'similarity': 0.47},
 {'title': 'Sunabouzu', 'similarity': 0.459},
 {'title': 'Kikou Ryouhei Merowlink', 'similarity': 0.445},
 {'title': 'Tetsuwan Birdy Decode:02', 'similarity': 0.44}]

In [99]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
#model = SentenceTransformer("all-mpnet-base-v2")

def search(query, top_k=5):
    vec = model.encode([query])
    vec = vec / np.linalg.norm(vec, axis=1, keepdims=True)

    scores, indices = index.search(vec, top_k)

    results = anime_df.iloc[indices[0]][["title", "score", "genres", "synopsis"]]
    results["similarity"] = scores[0]
    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [102]:
def search(query, top_k=10):
    vec = model.encode([query])
    vec = vec / np.linalg.norm(vec, axis=1, keepdims=True)

    scores, indices = index.search(vec, top_k)

    results = anime_df.iloc[indices[0]][["title", "score", "genres", "synopsis"]].copy()
    results["similarity"] = scores[0]

    results["score_norm"] = results["score"].fillna(5.0) / 10.0

    results["combined"] = (0.8 * results["similarity"]) + (0.2 * results["score_norm"])

    return results.sort_values("combined", ascending=False)


In [118]:
search("fantasy escapism")

,title,score,genres,synopsis,similarity,score_norm,combined
12080,Di Yi Xulie,7.64,"['Action', 'Adventure', 'Fantasy', 'Sci-Fi']",A new era begins in a devastated world gripped...,0.483699,0.764,0.539759
700,Tenshi no Tamago,7.71,"['Avant Garde', 'Drama', 'Fantasy']",The surrealist world of Tenshi no Tamago is de...,0.479323,0.771,0.537659
7624,Ying Xiong Bie Nao,6.09,"['Action', 'Fantasy']",The series takes place on a parallel world fil...,0.489633,0.609,0.513506
10478,Fairy Ranmaru: Anata no Kokoro Otasuke Shimasu,5.91,"['Action', 'Fantasy', 'Slice of Life']","In a world where negative emotions fester, fiv...",0.482133,0.591,0.503906
5142,Shinkai Densetsu Meremanoid,5.62,"['Adventure', 'Fantasy']",The story takes place in the underwater world ...,0.483080,0.562,0.498864
13520,Quanneng Gaoshou,NaN,"['Action', 'Fantasy']","""I came from the wilderness, raised by five in...",0.486219,0.500,0.488975
13432,Tiancai Julebu,NaN,"['Action', 'Adventure', 'Fantasy']",It uses a changing world line to make forward-...,0.482749,0.500,0.486199
13415,Lun Hui Leyuan,NaN,"['Action', 'Adventure', 'Fantasy']","Traversing countless worlds, reborn through en...",0.480111,0.500,0.484089
10400,Lalala Demacia,NaN,"['Action', 'Fantasy']",The script draws on various game culture that ...,0.471329,0.500,0.477063
10702,Joint,4.93,['Avant Garde'],An event that happens to a man based on growth...,0.472406,0.493,0.476525
